
# RT Notebook 16 — Projection / DoF Meaningful Experiment

**Campaign ID:** `MPF_SIM_PROJECTION_DOF_MEANINGFUL_001`

This notebook converts the earlier DoF-isolation specification into a bounded falsification experiment.

## Primary hypothesis

> Increasing degrees of freedom generate organizational structure beyond what is expected from graph size alone.

## Competing hypotheses

- **H₀ — combinatorial inflation:** apparent richness is explained by node/edge count and generic random structure.
- **H₁ — organizational enrichment:** lawful organizations differ from matched random controls after normalization.

## Required evidence

The notebook does **not** use raw counts as its main evidence. It computes:

1. matched random-graph controls;
2. normalized effect sizes and empirical p-values;
3. perturbation robustness;
4. projection retention and recoverability;
5. scaling-law comparisons;
6. counterexamples and null results.

## Claim boundary

Results are bounded to the supplied generated catalogs and this operational graph encoding. They do not establish external physical validity or a formal theorem.


In [ ]:

#@title 1. Configuration
from pathlib import Path

CAMPAIGN_ID = "MPF_SIM_PROJECTION_DOF_MEANINGFUL_001"
SEED = 160016
RANDOM_CONTROLS_PER_ORGANIZATION = 40   # Raise to 200+ for a publication-grade run
PERTURBATIONS_PER_ORGANIZATION = 30
BOOTSTRAP_REPEATS = 1000
MAX_ORGANIZATIONS_PER_DOF = None         # e.g. 100 for a faster test; None uses all
OUTPUT_DIR = Path("/content/rt_dof_meaningful_results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SOURCE_CANDIDATES = [
    Path("/content/MPF_SIM_PROJECTION_DOF_ISOLATION_001.zip"),
    Path("/mnt/data/MPF_SIM_PROJECTION_DOF_ISOLATION_001.zip"),
]

print("Campaign:", CAMPAIGN_ID)
print("Output directory:", OUTPUT_DIR)


In [ ]:

#@title 2. Imports and source archive
import ast
import io
import json
import math
import random
import re
import shutil
import statistics
import zipfile
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx

from scipy.stats import spearmanr
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

rng = np.random.default_rng(SEED)
random.seed(SEED)

source_zip = next((p for p in SOURCE_CANDIDATES if p.exists()), None)

if source_zip is None:
    try:
        from google.colab import files
        print("Upload MPF_SIM_PROJECTION_DOF_ISOLATION_001.zip")
        uploaded = files.upload()
        if not uploaded:
            raise FileNotFoundError("No archive uploaded.")
        name, data = next(iter(uploaded.items()))
        source_zip = Path("/content") / name
        source_zip.write_bytes(data)
    except ImportError as exc:
        raise FileNotFoundError(
            "Source archive not found. Place MPF_SIM_PROJECTION_DOF_ISOLATION_001.zip "
            "in /content or /mnt/data."
        ) from exc

EXTRACT_DIR = OUTPUT_DIR / "source"
if EXTRACT_DIR.exists():
    shutil.rmtree(EXTRACT_DIR)
EXTRACT_DIR.mkdir(parents=True)

with zipfile.ZipFile(source_zip) as zf:
    zf.extractall(EXTRACT_DIR)

source_manifest = json.loads((EXTRACT_DIR / "manifest.json").read_text())
print("Loaded:", source_zip)
print(json.dumps(source_manifest, indent=2))



## Operational representation

The catalog stores symbolic organization signatures rather than explicit graphs.  
For this experiment, each signature is parsed as an ordered rooted expression graph:

- every symbol or operator becomes a node;
- parent–argument relations become directed edges;
- repeated symbols remain distinct occurrences;
- graph statistics are computed on both directed and undirected views.

This encoding is explicit and falsifiable. The results are about this encoding, not all possible encodings.


In [ ]:

#@title 3. Parse symbolic signatures into expression graphs

TOKEN_RE = re.compile(r"[A-Za-z_ℰδΔα]+|[-+]?\d+(?:\.\d+)?|[^\s(),]+|[(),]")

def tokenize_signature(signature):
    return TOKEN_RE.findall(str(signature))

def parse_signature(signature):
    tokens = tokenize_signature(signature)
    pos = 0

    def parse_expr():
        nonlocal pos
        if pos >= len(tokens):
            raise ValueError("Unexpected end of signature")
        label = tokens[pos]
        pos += 1
        children = []
        if pos < len(tokens) and tokens[pos] == "(":
            pos += 1
            while pos < len(tokens) and tokens[pos] != ")":
                children.append(parse_expr())
                if pos < len(tokens) and tokens[pos] == ",":
                    pos += 1
                elif pos < len(tokens) and tokens[pos] != ")":
                    # Preserve unusual punctuation as an atomic child.
                    children.append((tokens[pos], []))
                    pos += 1
            if pos >= len(tokens) or tokens[pos] != ")":
                raise ValueError(f"Unbalanced signature: {signature}")
            pos += 1
        return (label, children)

    root = parse_expr()
    # Preserve trailing tokens as children of an explicit sequence root.
    if pos < len(tokens):
        tail = []
        while pos < len(tokens):
            if tokens[pos] not in {",", "(", ")"}:
                tail.append((tokens[pos], []))
            pos += 1
        root = ("SEQ", [root] + tail)
    return root

def tree_to_digraph(tree):
    G = nx.DiGraph()
    counter = 0

    def visit(item, parent=None, order=0):
        nonlocal counter
        label, children = item
        node_id = counter
        counter += 1
        G.add_node(node_id, label=str(label), sibling_order=order)
        if parent is not None:
            G.add_edge(parent, node_id, order=order)
        for i, child in enumerate(children):
            visit(child, node_id, i)
        return node_id

    root = visit(tree)
    G.graph["root"] = root
    return G

def signature_to_graph(signature):
    return tree_to_digraph(parse_signature(signature))

examples = ["P", "RT(P)", "RT(P,P)", "A(RT(P),E(P))"]
for s in examples:
    G = signature_to_graph(s)
    print(s, "nodes=", G.number_of_nodes(), "edges=", G.number_of_edges())


In [ ]:

#@title 4. Load and validate organization catalogs

catalog_rows = []
for path in sorted(EXTRACT_DIR.glob("lawful_catalog_dof_*.json")):
    dof = int(re.search(r"dof_(\d+)", path.name).group(1))
    records = json.loads(path.read_text())
    if MAX_ORGANIZATIONS_PER_DOF is not None:
        records = records[:MAX_ORGANIZATIONS_PER_DOF]
    for rec in records:
        rec = dict(rec)
        rec["source_file"] = path.name
        rec["dof"] = int(rec.get("dof", dof))
        catalog_rows.append(rec)

catalog = pd.DataFrame(catalog_rows)
required = {"dof", "signature", "depth", "symmetry_class", "closed", "stable",
            "primitive_preserved", "novel_to_lower_dof"}
missing = required - set(catalog.columns)
if missing:
    raise ValueError(f"Catalog missing required fields: {sorted(missing)}")

catalog["organization_id"] = [
    f"D{dof:02d}_{i:05d}" for i, dof in enumerate(catalog["dof"])
]
catalog["signature"] = catalog["signature"].astype(str)

print("Rows:", len(catalog))
display(catalog.groupby("dof").agg(
    organizations=("organization_id", "count"),
    closed_rate=("closed", "mean"),
    stable_rate=("stable", "mean"),
    novel_rate=("novel_to_lower_dof", "mean"),
))


In [ ]:

#@title 5. Generic graph feature extraction

def shannon_entropy(values):
    counts = np.array(list(Counter(values).values()), dtype=float)
    if counts.size == 0:
        return 0.0
    p = counts / counts.sum()
    return float(-(p * np.log2(p)).sum())

def graph_features(G):
    U = G.to_undirected()
    n = G.number_of_nodes()
    m = G.number_of_edges()
    root = G.graph.get("root", 0)

    if n == 0:
        return {}

    depths = nx.single_source_shortest_path_length(G, root)
    out_degrees = np.array([G.out_degree(v) for v in G.nodes()], dtype=float)
    undegrees = np.array([U.degree(v) for v in U.nodes()], dtype=float)
    labels = [G.nodes[v].get("label", "") for v in G.nodes()]

    articulation = list(nx.articulation_points(U)) if n > 1 else []
    bridges = list(nx.bridges(U)) if n > 1 else []
    components = nx.number_connected_components(U)
    cycle_rank = m - n + components

    return {
        "node_count_graph": n,
        "edge_count_graph": m,
        "density_graph": nx.density(G),
        "max_depth_graph": max(depths.values()) if depths else 0,
        "mean_depth_graph": float(np.mean(list(depths.values()))) if depths else 0.0,
        "leaf_fraction": float(np.mean(out_degrees == 0)),
        "mean_branching": float(out_degrees.mean()),
        "max_branching": float(out_degrees.max()),
        "degree_entropy": shannon_entropy(undegrees.astype(int)),
        "label_entropy": shannon_entropy(labels),
        "unique_label_ratio": len(set(labels)) / n,
        "articulation_count": len(articulation),
        "articulation_fraction": len(articulation) / n,
        "bridge_count": len(bridges),
        "bridge_fraction": len(bridges) / max(m, 1),
        "cycle_rank": cycle_rank,
        "weak_component_count": nx.number_weakly_connected_components(G),
        "scc_count": nx.number_strongly_connected_components(G),
        "transitivity": nx.transitivity(U) if n >= 3 else 0.0,
    }

feature_rows = []
graph_store = {}
parse_failures = []

for row in catalog.itertuples(index=False):
    try:
        G = signature_to_graph(row.signature)
        graph_store[row.organization_id] = G
        f = graph_features(G)
        f.update({
            "organization_id": row.organization_id,
            "dof": row.dof,
            "signature": row.signature,
            "catalog_depth": row.depth,
            "symmetry_class": row.symmetry_class,
            "closed": bool(row.closed),
            "stable": bool(row.stable),
            "primitive_preserved": bool(row.primitive_preserved),
            "novel_to_lower_dof": bool(row.novel_to_lower_dof),
        })
        feature_rows.append(f)
    except Exception as exc:
        parse_failures.append({"organization_id": row.organization_id,
                               "signature": row.signature, "error": repr(exc)})

features = pd.DataFrame(feature_rows)
print("Parsed:", len(features), "Failures:", len(parse_failures))
display(features.head())



## Matched null controls

Each lawful expression graph is compared with random controls matched on:

- node count;
- edge count;
- directedness.

For tree-like organizations, the primary control is a uniformly sampled random labeled tree with random root orientation.  
For denser graphs, additional edges are sampled without self-loops.

The control asks whether observed topology exceeds what graph size alone would produce.


In [ ]:

#@title 6. Matched random graph generator

def matched_random_graph(G, rng):
    n = G.number_of_nodes()
    m = G.number_of_edges()
    H = nx.DiGraph()
    H.add_nodes_from(range(n))
    if n <= 1:
        H.graph["root"] = 0
        return H

    # Start with a uniformly random undirected labeled tree.
    seed = int(rng.integers(0, 2**32 - 1))
    U = nx.random_labeled_tree(n, seed=seed)
    root = int(rng.integers(0, n))

    # Orient away from the root.
    for parent, child in nx.bfs_edges(U, root):
        H.add_edge(parent, child)

    # Add matched extra edges when the source has them.
    candidates = [(u, v) for u in range(n) for v in range(n)
                  if u != v and not H.has_edge(u, v)]
    rng.shuffle(candidates)
    for u, v in candidates[:max(0, m - (n - 1))]:
        H.add_edge(u, v)

    # Match label multiplicity without preserving organization.
    labels = [G.nodes[v].get("label", "") for v in G.nodes()]
    rng.shuffle(labels)
    for v, label in zip(H.nodes(), labels):
        H.nodes[v]["label"] = label
    H.graph["root"] = root
    return H

TEST_G = next(iter(graph_store.values()))
TEST_H = matched_random_graph(TEST_G, rng)
print(graph_features(TEST_G))
print(graph_features(TEST_H))


In [ ]:

#@title 7. Run matched controls and calculate normalized effects

METRICS = [
    "max_depth_graph", "mean_depth_graph", "leaf_fraction", "mean_branching",
    "max_branching", "degree_entropy", "label_entropy", "unique_label_ratio",
    "articulation_count", "articulation_fraction", "bridge_count",
    "bridge_fraction", "cycle_rank", "scc_count", "transitivity"
]

control_rows = []
effect_rows = []

for i, row in features.iterrows():
    G = graph_store[row["organization_id"]]
    null_records = []
    for rep in range(RANDOM_CONTROLS_PER_ORGANIZATION):
        H = matched_random_graph(G, rng)
        hf = graph_features(H)
        hf.update({
            "organization_id": row["organization_id"],
            "dof": int(row["dof"]),
            "replicate": rep,
        })
        null_records.append(hf)
        control_rows.append(hf)

    null_df = pd.DataFrame(null_records)
    effects = {
        "organization_id": row["organization_id"],
        "dof": int(row["dof"]),
        "signature": row["signature"],
        "novel_to_lower_dof": bool(row["novel_to_lower_dof"]),
        "closed": bool(row["closed"]),
        "stable": bool(row["stable"]),
    }

    for metric in METRICS:
        obs = float(row[metric])
        vals = null_df[metric].astype(float).to_numpy()
        mu = float(vals.mean())
        sd = float(vals.std(ddof=1))
        z = (obs - mu) / sd if sd > 0 else (0.0 if obs == mu else np.sign(obs - mu) * np.inf)
        # Two-sided finite-sample empirical p-value.
        dev_obs = abs(obs - mu)
        dev_null = np.abs(vals - mu)
        p = (1 + int(np.sum(dev_null >= dev_obs))) / (len(vals) + 1)
        effects[f"{metric}__observed"] = obs
        effects[f"{metric}__null_mean"] = mu
        effects[f"{metric}__z"] = z
        effects[f"{metric}__p_empirical"] = p

    effect_rows.append(effects)

controls = pd.DataFrame(control_rows)
effects = pd.DataFrame(effect_rows)
print("Control graphs:", len(controls))
display(effects.head())


In [ ]:

#@title 8. Aggregate organizational enrichment by DoF

z_columns = [c for c in effects.columns if c.endswith("__z")]
finite_effects = effects.replace([np.inf, -np.inf], np.nan)
finite_effects["organization_enrichment_index"] = (
    finite_effects[z_columns].abs().mean(axis=1, skipna=True)
)

dof_summary = finite_effects.groupby("dof").agg(
    organizations=("organization_id", "count"),
    enrichment_mean=("organization_enrichment_index", "mean"),
    enrichment_median=("organization_enrichment_index", "median"),
    enrichment_std=("organization_enrichment_index", "std"),
    novel_rate=("novel_to_lower_dof", "mean"),
    closed_rate=("closed", "mean"),
    stable_rate=("stable", "mean"),
).reset_index()

display(dof_summary)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(dof_summary["dof"], dof_summary["enrichment_mean"], marker="o")
ax.set_xlabel("Degrees of freedom")
ax.set_ylabel("Mean absolute matched-null z-score")
ax.set_title("Organizational enrichment beyond graph-size controls")
ax.grid(True, alpha=0.3)
plt.show()


In [ ]:

#@title 9. Bootstrap trend test: does normalized organization rise with DoF?

def bootstrap_spearman(df, x, y, repeats=1000, seed=SEED):
    local_rng = np.random.default_rng(seed)
    vals = []
    work = df[[x, y]].dropna()
    for _ in range(repeats):
        sample = work.iloc[local_rng.integers(0, len(work), len(work))]
        rho, _ = spearmanr(sample[x], sample[y])
        vals.append(rho)
    vals = np.asarray(vals, dtype=float)
    return {
        "rho_observed": float(spearmanr(work[x], work[y]).statistic),
        "ci_2.5": float(np.nanpercentile(vals, 2.5)),
        "ci_97.5": float(np.nanpercentile(vals, 97.5)),
        "p_rho_le_zero": float(np.mean(vals <= 0)),
    }

trend_result = bootstrap_spearman(
    finite_effects, "dof", "organization_enrichment_index",
    repeats=BOOTSTRAP_REPEATS
)
print(json.dumps(trend_result, indent=2))



## Perturbation robustness

A meaningful organization should not be judged only by its ideal form.

For each graph, the notebook performs single-edge deletion or rewiring and measures retention of its generic feature vector:

\[
R = 1 - \frac{\lVert f(G)-f(G')\rVert_1}
{\lVert f(G)\rVert_1+\lVert f(G')\rVert_1+\epsilon}
\]

Higher values indicate stronger structural retention under local perturbation.


In [ ]:

#@title 10. Perturbation experiment

ROBUSTNESS_METRICS = [
    "max_depth_graph", "leaf_fraction", "mean_branching",
    "degree_entropy", "articulation_fraction", "bridge_fraction",
    "cycle_rank", "transitivity"
]

def perturb_graph(G, rng):
    H = G.copy()
    H.graph.update(G.graph)
    edges = list(H.edges())
    nodes = list(H.nodes())
    if not edges or len(nodes) < 2:
        return H, "no_op"

    mode = rng.choice(["delete", "rewire"])
    edge = edges[int(rng.integers(0, len(edges)))]
    H.remove_edge(*edge)

    if mode == "rewire":
        candidates = [(u, v) for u in nodes for v in nodes
                      if u != v and not H.has_edge(u, v)]
        if candidates:
            new_edge = candidates[int(rng.integers(0, len(candidates)))]
            H.add_edge(*new_edge)
        else:
            mode = "delete"
    return H, mode

def feature_retention(a, b, keys=ROBUSTNESS_METRICS):
    av = np.array([float(a[k]) for k in keys])
    bv = np.array([float(b[k]) for k in keys])
    return float(1.0 - np.abs(av - bv).sum() /
                 (np.abs(av).sum() + np.abs(bv).sum() + 1e-12))

perturbation_rows = []
for row in features.itertuples(index=False):
    G = graph_store[row.organization_id]
    original = graph_features(G)
    for rep in range(PERTURBATIONS_PER_ORGANIZATION):
        H, mode = perturb_graph(G, rng)
        changed = graph_features(H)
        perturbation_rows.append({
            "organization_id": row.organization_id,
            "dof": int(row.dof),
            "replicate": rep,
            "mode": mode,
            "feature_retention": feature_retention(original, changed),
            "connected_after": (
                nx.is_weakly_connected(H) if H.number_of_nodes() > 0 else True
            ),
        })

perturbations = pd.DataFrame(perturbation_rows)
robustness_summary = perturbations.groupby("dof").agg(
    mean_retention=("feature_retention", "mean"),
    median_retention=("feature_retention", "median"),
    disconnection_rate=("connected_after", lambda s: 1 - s.mean()),
).reset_index()

display(robustness_summary)


In [ ]:

#@title 11. Analyze the supplied projection-loss table

projection_path = EXTRACT_DIR / "projection_loss_table.csv"
projection = pd.read_csv(projection_path)

print("Columns:", list(projection.columns))
display(projection.head())

# Normalize common boolean spellings.
for col in projection.columns:
    if projection[col].dtype == object:
        lowered = projection[col].astype(str).str.lower()
        if lowered.isin(["true", "false"]).all():
            projection[col] = lowered.eq("true")

numeric_cols = projection.select_dtypes(include=[np.number]).columns.tolist()
boolean_cols = projection.select_dtypes(include=["bool"]).columns.tolist()

projection_summary = {
    "rows": int(len(projection)),
    "numeric_columns": numeric_cols,
    "boolean_columns": boolean_cols,
}

# Use source-target gap when available.
source_candidates = [c for c in projection.columns if c.lower() in {"source_dof", "from_dof"}]
target_candidates = [c for c in projection.columns if c.lower() in {"target_dof", "to_dof"}]
if source_candidates and target_candidates:
    source_col, target_col = source_candidates[0], target_candidates[0]
    projection["dof_gap"] = projection[source_col] - projection[target_col]
    agg_map = {}
    for c in numeric_cols:
        if c not in {source_col, target_col}:
            agg_map[c] = "mean"
    for c in boolean_cols:
        agg_map[c] = "mean"
    if agg_map:
        projection_by_gap = projection.groupby("dof_gap").agg(agg_map).reset_index()
        display(projection_by_gap)
else:
    projection_by_gap = pd.DataFrame()

print(json.dumps(projection_summary, indent=2))


In [ ]:

#@title 12. Scaling-law comparison

# Compare models on normalized enrichment, not raw organization count.
scaling_data = dof_summary[["dof", "enrichment_mean"]].dropna().copy()
x = scaling_data["dof"].to_numpy(dtype=float)
y = scaling_data["enrichment_mean"].to_numpy(dtype=float)

def fit_transform(name, transform):
    X = transform(x).reshape(-1, 1)
    model = LinearRegression().fit(X, y)
    pred = model.predict(X)
    return {
        "model": name,
        "r2": float(r2_score(y, pred)),
        "intercept": float(model.intercept_),
        "coefficient": float(model.coef_[0]),
        "prediction": pred.tolist(),
    }

model_results = [
    fit_transform("linear", lambda z: z),
    fit_transform("logarithmic", lambda z: np.log(z)),
    fit_transform("power_loglog", lambda z: np.log(z)),
]

# For power law, model log(y) against log(x) only when y > 0.
if np.all(y > 0):
    model = LinearRegression().fit(np.log(x).reshape(-1, 1), np.log(y))
    pred = np.exp(model.predict(np.log(x).reshape(-1, 1)))
    model_results[-1] = {
        "model": "power_law",
        "r2": float(r2_score(y, pred)),
        "intercept_log": float(model.intercept_),
        "exponent": float(model.coef_[0]),
        "prediction": pred.tolist(),
    }

scaling_models = pd.DataFrame([{k: v for k, v in r.items() if k != "prediction"}
                               for r in model_results]).sort_values("r2", ascending=False)
display(scaling_models)


In [ ]:

#@title 13. Counterexamples and falsification conditions

median_enrichment = finite_effects.groupby("dof")["organization_enrichment_index"].median()
finite_effects["dof_median_enrichment"] = finite_effects["dof"].map(median_enrichment)

counterexamples = finite_effects[
    (finite_effects["dof"] >= finite_effects["dof"].median()) &
    (finite_effects["organization_enrichment_index"] <
     finite_effects["dof_median_enrichment"])
].copy()

# Stronger falsification candidate: higher DoF but below the maximum enrichment at lower DoF.
lower_max = {}
running = -np.inf
for dof in sorted(finite_effects["dof"].unique()):
    lower_max[dof] = running
    vals = finite_effects.loc[
        finite_effects["dof"] == dof, "organization_enrichment_index"
    ].dropna()
    if len(vals):
        running = max(running, vals.max())

finite_effects["lower_dof_max_enrichment"] = finite_effects["dof"].map(lower_max)
strong_counterexamples = finite_effects[
    finite_effects["organization_enrichment_index"] <
    finite_effects["lower_dof_max_enrichment"]
].copy()

print("Within-DoF low-enrichment cases:", len(counterexamples))
print("Higher-DoF below a lower-DoF maximum:", len(strong_counterexamples))
display(strong_counterexamples[
    ["organization_id", "dof", "signature", "organization_enrichment_index",
     "lower_dof_max_enrichment"]
].head(20))


In [ ]:

#@title 14. Decision protocol

rho = trend_result["rho_observed"]
ci_low = trend_result["ci_2.5"]
p_nonpositive = trend_result["p_rho_le_zero"]

if ci_low > 0 and p_nonpositive < 0.05:
    decision = "SUPPORT_H1_BOUNDED"
    interpretation = (
        "Normalized organizational enrichment rises with DoF in this bounded "
        "catalog and operational graph encoding."
    )
elif trend_result["ci_97.5"] < 0:
    decision = "SUPPORT_NEGATIVE_TREND"
    interpretation = (
        "Normalized organizational enrichment falls with DoF in this bounded domain."
    )
else:
    decision = "FAIL_TO_REJECT_H0"
    interpretation = (
        "The experiment does not establish increasing organization beyond "
        "matched graph-size controls."
    )

decision_record = {
    "campaign_id": CAMPAIGN_ID,
    "decision": decision,
    "interpretation": interpretation,
    "trend": trend_result,
    "claim_ceiling": "C2_BOUNDED_NOTEBOOK_OUTPUT_AFTER_GOVERNED_INDUCTION",
    "non_claims": [
        "No formal theorem.",
        "No implementation-independent conclusion.",
        "No external physical validation.",
        "No claim that DoF universally generates organization."
    ]
}
print(json.dumps(decision_record, indent=2))


In [ ]:

#@title 15. Export governed result bundle

import hashlib
from datetime import datetime, timezone

def write_json(path, obj):
    Path(path).write_text(json.dumps(obj, indent=2, default=str))

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest().upper()

features.to_csv(OUTPUT_DIR / "organization_features.csv", index=False)
controls.to_csv(OUTPUT_DIR / "matched_random_controls.csv", index=False)
effects.to_csv(OUTPUT_DIR / "normalized_effects.csv", index=False)
dof_summary.to_csv(OUTPUT_DIR / "dof_summary.csv", index=False)
perturbations.to_csv(OUTPUT_DIR / "perturbation_trials.csv", index=False)
robustness_summary.to_csv(OUTPUT_DIR / "robustness_summary.csv", index=False)
projection.to_csv(OUTPUT_DIR / "projection_loss_analyzed.csv", index=False)
scaling_models.to_csv(OUTPUT_DIR / "scaling_model_comparison.csv", index=False)
strong_counterexamples.to_csv(OUTPUT_DIR / "counterexamples.csv", index=False)

write_json(OUTPUT_DIR / "parse_failures.json", parse_failures)
write_json(OUTPUT_DIR / "decision_record.json", decision_record)

result_files = [
    p for p in OUTPUT_DIR.iterdir()
    if p.is_file() and p.name not in {"manifest.json"}
]

manifest = {
    "campaign_id": CAMPAIGN_ID,
    "status": "EXECUTED",
    "created_at": datetime.now(timezone.utc).isoformat(),
    "seed": SEED,
    "source_archive": source_zip.name,
    "source_manifest_spec_id": source_manifest.get("spec_id"),
    "rows_loaded": int(len(catalog)),
    "rows_parsed": int(len(features)),
    "parse_failure_count": int(len(parse_failures)),
    "random_controls_per_organization": RANDOM_CONTROLS_PER_ORGANIZATION,
    "perturbations_per_organization": PERTURBATIONS_PER_ORGANIZATION,
    "bootstrap_repeats": BOOTSTRAP_REPEATS,
    "operational_representation": (
        "Ordered rooted expression graph parsed from catalog signature."
    ),
    "decision": decision,
    "claim_ceiling": "C2_BOUNDED_NOTEBOOK_OUTPUT_AFTER_GOVERNED_INDUCTION",
    "output_files": {}
}

for path in result_files:
    manifest["output_files"][path.name] = {
        "sha256": sha256(path),
        "bytes": path.stat().st_size,
    }

write_json(OUTPUT_DIR / "manifest.json", manifest)

bundle_path = Path("/content") / f"{CAMPAIGN_ID}_RESULTS.zip"
if bundle_path.exists():
    bundle_path.unlink()
with zipfile.ZipFile(bundle_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for path in sorted(OUTPUT_DIR.iterdir()):
        if path.is_file():
            zf.write(path, arcname=path.name)

print("Result bundle:", bundle_path)
print("SHA-256:", sha256(bundle_path))
print(json.dumps(manifest, indent=2)[:4000])


In [ ]:

#@title 16. Download results (Colab only)
try:
    from google.colab import files
    files.download(str(bundle_path))
except ImportError:
    print("Not running in Colab. Result bundle saved at:", bundle_path)



## Interpretation protocol

### Result A — normalized enrichment rises with DoF

Interpretation:

> Within this bounded catalog and graph encoding, organizational structure changes with DoF beyond matched graph-size expectations.

This does not establish a universal law. Next require independent generator replication.

### Result B — raw counts rise but normalized enrichment does not

Interpretation:

> The apparent DoF effect is consistent with combinatorial inflation.

This is a successful falsification of the stronger organizational-enrichment claim.

### Result C — only selected metrics rise

Interpretation:

> DoF does not produce a single scalar increase in organization; it changes particular structural dimensions.

Investigate the surviving metrics separately.

### Result D — perturbation robustness falls as DoF rises

Interpretation:

> Higher-DoF organizations may be richer but more fragile.

Richness and stability must then be treated as distinct quantities.

### Result E — projection preservation remains high despite information loss

Interpretation:

> Projection may preserve organizational class while discarding implementation detail.

This motivates a separate reconstruction experiment.
